# Адаптированная версия для локального PDF и локальных моделей

Этот ноутбук адаптирован для работы с:
- Локальным PDF файлом: `data/raw/instr.pdf`
- Локальными моделями embeddings через API или напрямую

## Запуск через Docker

```bash
docker-compose up -d
```

Затем откройте в браузере ссылку из логов (токен в URL):
```bash
docker logs llm_notebook
```
Или: **http://localhost:8888** (скопируйте token из логов).

В Docker ноутбук автоматически использует API по адресу `http://embeddings_service:8080`.

## Локальный запуск

1. Запустите `docker-compose up -d` (embeddings + ollama).
2. Откройте ноутбук в Jupyter локально; будет использоваться `http://localhost:8080`.

In [5]:
import os, json, time
import requests
from typing import List, Dict, Any

In [4]:
import pdfplumber
import re
import numpy as np
import pandas as pd
import os
import sys
from IPython.display import display

# Путь к модулю local_embeddings (работает и в Docker, и локально)
sys.path.insert(0, os.getcwd())
from local_embeddings import get_embedding_model, LocalEmbeddings

# Настройки
# В Docker: EMBEDDINGS_API_URL=http://embeddings_service:8080
# Локально: localhost:8080
USE_API = True
# API_URL = os.environ.get("EMBEDDINGS_API_URL", "http://localhost:8080")
API_URL = "http://localhost:8081"
PDF_PATH = "data/raw/instr.pdf"  # Локальный путь к PDF (Cookiecutter: raw)

# Проверка наличия PDF
if not os.path.exists(PDF_PATH):
    raise FileNotFoundError(f"PDF файл не найден: {PDF_PATH}")
    
print(f"✓ PDF файл найден: {PDF_PATH}")
print(f"✓ Режим работы: {'API' if USE_API else 'Прямой'}")

ModuleNotFoundError: No module named 'local_embeddings'

In [3]:
# Парсинг PDF
pdf_path = PDF_PATH

# pages_to_parse = [18, 23]   # страницы таблиц
pages_to_parse = range(7, 24)  # Адаптируйте под ваш PDF

all_tables = []
all_text = []

def words_to_lines(words, y_tol=3.0):
    if not words:
        return []
    words = sorted(words, key=lambda w: (w["top"], w["x0"]))
    lines, cur, cur_top = [], [], words[0]["top"]

    for w in words:
        if abs(w["top"] - cur_top) <= y_tol:
            cur.append(w)
        else:
            cur = sorted(cur, key=lambda x: x["x0"])
            lines.append(" ".join(x["text"] for x in cur))
            cur = [w]
            cur_top = w["top"]

    cur = sorted(cur, key=lambda x: x["x0"])
    lines.append(" ".join(x["text"] for x in cur))
    return lines

def clean(text: str) -> str:
    text = text.replace("\u00ad", "")  # soft hyphen
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def inside(w, bb, pad=1.5):
    x0, top, x1, bottom = bb
    cx = (w["x0"] + w["x1"]) / 2
    cy = (w["top"] + w["bottom"]) / 2
    return (x0-pad) <= cx <= (x1+pad) and (top-pad) <= cy <= (bottom+pad)

with pdfplumber.open(pdf_path) as pdf:
    for page_idx in pages_to_parse:
        if page_idx >= len(pdf.pages):
            print(f"Пропуск страницы {page_idx} (всего страниц: {len(pdf.pages)})")
            continue
            
        page = pdf.pages[page_idx]

        # ---- 1. Ищем таблицы ----
        tables = page.find_tables()
        if tables:
            for t in tables:
                all_tables.append(t.extract())

        # ---- 2. Извлекаем текст ----
        words = page.extract_words()
        if not words:
            continue

        # Находим bounding boxes таблиц
        table_bboxes = [t.bbox for t in tables] if tables else []

        # Фильтруем слова вне таблиц
        text_words = [w for w in words if not any(inside(w, bb) for bb in table_bboxes)]

        if text_words:
            lines = words_to_lines(text_words)
            page_text = "\n".join(lines)
            page_text = clean(page_text)
            if page_text.strip():
                all_text.append(page_text)

print(f"Извлечено страниц: {len(all_text)}")
print(f"Найдено таблиц: {len(all_tables)}")

NameError: name 'PDF_PATH' is not defined

In [21]:
# Обработка текста документа
doc_text = "\n\n".join(p.strip() for p in all_text if p and p.strip())

# Удаление номеров листов и служебной информации
doc_text = re.sub(r"Лист\s+\d+\s+Листов\s+\d+", "", doc_text)
doc_text = re.sub(r"ТИ\s*305\.2-01-2022", "", doc_text)
doc_text = re.sub(r"\n{3,}", "\n\n", doc_text).strip()
doc_text = re.sub(r"Рисунок\s+\d+.*", "", doc_text)

print(f"Длина документа: {len(doc_text)} символов")
print(f"Первые 500 символов:\n{doc_text[:500]}")

Длина документа: 32879 символов
Первые 500 символов:
Введение
Настоящая технологическая инструкция регламентирует
последовательность технологических операций приготовления никелевого
католита, правила ведения и управления процессом, содержит характеристику
сырья, материалов, оборудования и товарной продукции, методы контроля и
метрологическое обеспечение, а также требования охраны труда и
промышленной безопасности.
Инструкция разработана в соответствии с проектом ФМ.04450-П-ИОС7.1-С
«ЦЭН. ОЭН-2. Электроэкстракция никеля из растворов хлорного раств


In [22]:
# Chunking документа
HEADER_RE = r"\n(?=(?:\d+(?:\.\d+)*|\d+)\s+[А-ЯA-ZЁ])"

def chunk_by_headers(text, min_chars=250):
    parts = re.split(HEADER_RE, "\n" + text)
    return [p.strip() for p in parts if len(p.strip()) >= min_chars]

doc_chunks = chunk_by_headers(doc_text, min_chars=250)
print(f"Создано чанков: {len(doc_chunks)}")
print(f"Пример первого чанка:\n{doc_chunks[0][:300]}...")

Создано чанков: 19
Пример первого чанка:
Введение
Настоящая технологическая инструкция регламентирует
последовательность технологических операций приготовления никелевого
католита, правила ведения и управления процессом, содержит характеристику
сырья, материалов, оборудования и товарной продукции, методы контроля и
метрологическое обеспече...


In [23]:
# Вопросы для тестирования
questions = [
    "Что регламентирует технологическая инструкция ТИ 305.2-01-2022?",
    "Какова основная задача ГМО-2?",
    "Что такое католит согласно тексту?",
    "Что такое анолит согласно тексту?",
    "Из каких операций состоит производство католита?",
    "Какой химический состав исходного раствора ОРиД указан в инструкции?",
    "Какая концентрация и плотность серной кислоты используются и куда она сливается?",
    "Как описан хлор: его свойства, подача и хранение?",
    "Какова роль борной кислоты и как она вводится?",
    "Какие основные параметры и ограничения режима у пачука?"
]

# Золотые стандарты (индексы правильных чанков для каждого вопроса)
gold_manual = {
    1: 0,   # Введение
    2: 0,
    3: 0,
    4: 0,
    5: 0,
    6: 1,   # 1.1 Сырьё
    7: 2,   # 1.2.1 Серная кислота
    8: 4,   # 1.2.3 Хлор
    9: 5,   # 1.2.4 Борная кислота
    10: 12  # 3.1 Пачук
}

In [24]:
# Функции для оценки качества поиска
def build_semantic_gold(answers_for_gold, doc_emb, model):
    """Создание семантических золотых стандартов"""
    a_emb = model.encode(answers_for_gold, normalize_embeddings=True, show_progress_bar=False).astype(np.float32)
    gold = {}
    for qid in range(len(answers_for_gold)):
        scores = (doc_emb @ a_emb[qid].reshape(-1, 1)).ravel()
        gold[qid+1] = int(np.argmax(scores))
    return gold

def eval_vs_manual_gold(questions, doc_chunks, doc_emb, model,
                        gold_manual, ks=(1,3,5), qtext=lambda x:x):
    """Оценка качества поиска по золотым стандартам"""
    max_k = max(ks)
    hit = {k: 0 for k in ks}
    mrr_sum = 0.0

    for qid, q in enumerate(questions, start=1):
        g = gold_manual[qid]

        q_emb = model.encode([qtext(q)],
                             normalize_embeddings=True,
                             show_progress_bar=False).astype(np.float32)

        scores = (doc_emb @ q_emb.T).ravel()
        idx = np.argsort(-scores)[:max_k]
        top_ids = [int(i) for i in idx]

        for k in ks:
            hit[k] += int(g in top_ids[:k])

        if g in top_ids:
            mrr_sum += 1.0 / (top_ids.index(g) + 1)

    recalls = {f"Recall@{k}": hit[k]/len(questions) for k in ks}
    mrr = mrr_sum / len(questions)
    return recalls, mrr

In [25]:
# Тестирование различных моделей
models_to_try = [
    "BAAI/bge-m3",  # Лучший выбор для технической документации
    "intfloat/multilingual-e5-large",
    "intfloat/multilingual-e5-base",
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
]

results = []

for MODEL_NAME in models_to_try:
    print(f"\n{'='*60}")
    print(f"Тестирование модели: {MODEL_NAME}")
    print(f"{'='*60}")
    
    try:
        # Создаём модель (API или прямая)
        model = get_embedding_model(
            model_name=MODEL_NAME,
            use_api=USE_API,
            api_url=API_URL
        )
        
        # Функции для форматирования текста (для multilingual-e5)
        def qtext(q):
            return ("query: " + q) if "multilingual-e5" in MODEL_NAME else q

        def ptext(p):
            return ("passage: " + p) if "multilingual-e5" in MODEL_NAME else p

        # Генерация embeddings для документа
        print("Генерация embeddings для документа...")
        doc_emb = model.encode(
            [ptext(c) for c in doc_chunks],
            normalize_embeddings=True,
            show_progress_bar=True
        ).astype(np.float32)

        # Оценка качества
        print("Оценка качества поиска...")
        recalls, mrr = eval_vs_manual_gold(
            questions,
            doc_chunks,
            doc_emb,
            model,
            gold_manual,
            ks=(1,3,5),
            qtext=qtext
        )

        results.append({
            "model": MODEL_NAME,
            **recalls,
            "MRR@5": mrr
        })
        
        print(f"Результаты: {recalls}, MRR: {mrr:.3f}")
        
    except Exception as e:
        print(f"Ошибка при тестировании {MODEL_NAME}: {e}")
        import traceback
        traceback.print_exc()

df_models = pd.DataFrame(results).sort_values("Recall@1", ascending=False)
print("\n" + "="*60)
print("ИТОГОВЫЕ РЕЗУЛЬТАТЫ:")
print("="*60)
display(df_models)


Тестирование модели: BAAI/bge-m3
Предупреждение: Не удалось подключиться к API (HTTPConnectionPool(host='localhost', port=8081): Max retries exceeded with url: /health (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8081): Failed to establish a new connection: [Errno 111] Connection refused")))
Переключение на прямой режим...
Ошибка при тестировании BAAI/bge-m3: No module named 'sentence_transformers'

Тестирование модели: intfloat/multilingual-e5-large
Предупреждение: Не удалось подключиться к API (HTTPConnectionPool(host='localhost', port=8081): Max retries exceeded with url: /health (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8081): Failed to establish a new connection: [Errno 111] Connection refused")))
Переключение на прямой режим...
Ошибка при тестировании intfloat/multilingual-e5-large: No module named 'sentence_transformers'

Тестирование модели: intfloat/multilingual-e5-base
Предупреждение: Не удалось подключиться к API (HTTPConne

Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/urllib3/connection.py", line 204, in _new_conn
    sock = connection.create_connection(
  File "/usr/local/lib/python3.10/dist-packages/urllib3/util/connection.py", line 85, in create_connection
    raise err
  File "/usr/local/lib/python3.10/dist-packages/urllib3/util/connection.py", line 73, in create_connection
    sock.connect(sa)
ConnectionRefusedError: [Errno 111] Connection refused

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/urllib3/connectionpool.py", line 787, in urlopen
    response = self._make_request(
  File "/usr/local/lib/python3.10/dist-packages/urllib3/connectionpool.py", line 493, in _make_request
    conn.request(
  File "/usr/local/lib/python3.10/dist-packages/urllib3/connection.py", line 500, in request
    self.endheaders()
  File "/usr/lib/python3.10/http/client.py", line 1

KeyError: 'Recall@1'

In [ ]:
# Использование лучшей модели для сложных вопросов
MODEL_NAME = "BAAI/bge-m3"  # Лучший выбор для технической документации

print(f"Использование модели: {MODEL_NAME}")
model = get_embedding_model(
    model_name=MODEL_NAME,
    use_api=USE_API,
    api_url=API_URL
)

# Генерация embeddings
doc_emb = model.encode(
    doc_chunks,
    normalize_embeddings=True,
    show_progress_bar=True
).astype(np.float32)

# Сложные вопросы
questions_hard = [
    "Какие аспекты производственного процесса описывает данный регламент?",
    "Для какой цели функционирует гидрометаллургическое отделение №2?",
    "Как называется очищенный электролит, поступающий в электролиз?",
    "Какой раствор образуется при растворении НПТП в реакторах ОРиД?",
    "Перечисли технологические стадии подготовки электролита перед электролизом.",
    "Какие ограничения по концентрациям компонентов установлены для исходного раствора?",
    "Какие параметры качества и хранения предъявляются к концентрированной H2SO4?",
    "Какие физические свойства и схема подачи хлора предусмотрены технологией?",
    "Как обеспечивается стабилизация кислотности в прикатодной зоне?",
    "Какие конструктивные и режимные параметры характерны для реакционной ёмкости 170 м3?"
]

recalls, mrr = eval_vs_manual_gold(
    questions_hard,
    doc_chunks,
    doc_emb,
    model,
    gold_manual,
    ks=(1,3,5)
)

print(f"\nСложные вопросы - Результаты:")
print(f"Recall@1: {recalls['Recall@1']:.3f}")
print(f"Recall@3: {recalls['Recall@3']:.3f}")
print(f"Recall@5: {recalls['Recall@5']:.3f}")
print(f"MRR: {mrr:.3f}")

In [ ]:
# Пример поиска по запросу
def search_document(query, model, doc_chunks, doc_emb, top_k=3):
    """Поиск релевантных фрагментов документа"""
    query_emb = model.encode(
        [query],
        normalize_embeddings=True,
        show_progress_bar=False
    ).astype(np.float32)
    
    scores = (doc_emb @ query_emb.T).ravel()
    top_indices = np.argsort(-scores)[:top_k]
    
    results = []
    for idx in top_indices:
        results.append({
            'chunk_idx': int(idx),
            'score': float(scores[idx]),
            'text': doc_chunks[idx][:500] + '...' if len(doc_chunks[idx]) > 500 else doc_chunks[idx]
        })
    
    return results

# Тестовый запрос
test_query = "Что такое католит и как он производится?"
results = search_document(test_query, model, doc_chunks, doc_emb, top_k=3)

print(f"Запрос: {test_query}\n")
print("Топ-3 релевантных фрагмента:")
for i, r in enumerate(results, 1):
    print(f"\n{i}. Чанк #{r['chunk_idx']}, Score: {r['score']:.4f}")
    print(f"   {r['text']}")